# 🏠 Egypt Real Estate Appraiser — Final Pipeline
### Dubizzle.com.eg Scrape · 74,240 Records · Stratified ML Models

---

| Item | Value |
|------|-------|
| **Dataset** | Dubizzle.com.eg scrape (107,276 URLs → 74,240 clean records) |
| **Approach** | Stratified: one model per property type |
| **Best Models** | Random Forest (Apartment, Villa, Studio) + Gradient Boosting (Town House, Twin House, Duplex, Penthouse) |
| **Global R²** | 0.7576 |
| **Global MAE** | 3.532M EGP |
| **Global MAPE** | 27.87% |

**New features vs. previous datasets:**
- ✅ Ownership (Primary / Resale)
- ✅ Payment Option (Cash / Installment)
- ✅ Completion Status (Ready / Off-plan)
- ✅ City + District bounded location (1,473 combinations)


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib, os
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
print('✅ All libraries loaded')

## 2. Load Data & Apply Fixes

**Three fixes applied:**
1. `Status` filter — keep only OK listings (drop HTTP errors, timeouts)
2. Property Type — 2 rows had compound names leaked → filtered to valid types
3. District fix — missing → Compound → City; District==City → Compound


In [ ]:
df = pd.read_excel('dubizzle_listings.xlsx')
df.columns = [c.strip() for c in df.columns]
print(f'Raw records: {len(df):,}')
print(f'Status breakdown:\n{df["Status"].value_counts().head(5).to_string()}')

In [ ]:
# ── Fix 1: Keep only successfully scraped listings
df = df[df['Status'] == 'OK'].copy()

# ── Fix 2: Property type — remove 2 rows with compound names leaked
VALID_TYPES = ['Apartment','Stand Alone Villa','Town House','Twin House',
               'Duplex','Penthouse','Studio','iVilla','Hotel Apartment','Roof']
df = df[df['Property Type'].isin(VALID_TYPES)].copy()

# ── Strip whitespace from string columns
for c in ['Property Type','City','Governorate','District','Compound',
          'Ownership','Payment Option','Completion Status']:
    df[c] = df[c].astype(str).str.strip().replace('nan', np.nan)

print(f'After status + type filter: {len(df):,}')
df.head(3)

In [ ]:
# ── Fix 3: District fix — bounded location hierarchy
df['district_fixed'] = df['District'].copy()

# Missing District → use Compound
mask = df['district_fixed'].isna()
df.loc[mask, 'district_fixed'] = df.loc[mask, 'Compound']

# Still missing → use City
mask = df['district_fixed'].isna()
df.loc[mask, 'district_fixed'] = df.loc[mask, 'City']

# District == City → use Compound if available (more specific)
same_as_city = df['district_fixed'].str.strip() == df['City'].str.strip()
has_compound  = df['Compound'].notna()
df.loc[same_as_city & has_compound, 'district_fixed'] = \
    df.loc[same_as_city & has_compound, 'Compound']

# Create bounded location features
df['gov_city']      = df['Governorate'].str.strip() + ' — ' + df['City'].str.strip()
df['city_district'] = df['City'].str.strip() + ' / ' + df['district_fixed'].str.strip()

print(f'gov_city combinations    : {df["gov_city"].nunique()}')
print(f'city_district combinations: {df["city_district"].nunique()}')

## 3. Data Cleaning

In [ ]:
# Normalize property types
TYPE_MAP = {
    'Apartment':'Apartment', 'Stand Alone Villa':'Villa',
    'Town House':'Town House', 'Duplex':'Duplex', 'Twin House':'Twin House',
    'Penthouse':'Penthouse', 'Studio':'Studio', 'iVilla':'Villa',
    'Hotel Apartment':'Other', 'Roof':'Other',
}
df['prop_type'] = df['Property Type'].map(TYPE_MAP).fillna('Other')

# Price in millions
df['price_m'] = df['Price (EGP)'] / 1_000_000

# Remove outliers
p_hi = df['price_m'].quantile(0.995)
a_hi = df['Built-Up Area (m²)'].quantile(0.995)
n_before = len(df)
df = df[(df['price_m'] >= 0.5) & (df['price_m'] <= p_hi)]
df = df[(df['Built-Up Area (m²)'] >= 20) & (df['Built-Up Area (m²)'] <= a_hi)]
df = df[df['Bedrooms'].between(1, 8)]

# Drop duplicates
df = df.drop_duplicates(subset=['Price (EGP)','Built-Up Area (m²)','Bedrooms','City','district_fixed'])
df = df.dropna(subset=['price_m','Built-Up Area (m²)','Bedrooms','Governorate','City'])

print(f'Removed : {n_before - len(df):,} rows (outliers + duplicates)')
print(f'Clean   : {len(df):,} records')
print(f'Price   : {df["price_m"].min():.1f}M – {df["price_m"].max():.1f}M EGP')
print(f'Median  : {df["price_m"].median():.2f}M  |  Mean: {df["price_m"].mean():.2f}M')
print()
print('Property type distribution:')
print(df['prop_type'].value_counts().to_string())

## 4. Encode New Features

Three new binary features not available in the Kaggle dataset:
- **Ownership**: Primary (developer sale) vs Resale
- **Payment Option**: Cash vs Installment  
- **Completion Status**: Ready (immediate) vs Off-plan (under construction)


In [ ]:
df['ownership_enc']  = df['Ownership'].map({'Primary':1,'Resale':0}).fillna(0.5)
df['payment_enc']    = df['Payment Option'].map({'Cash':1,'Installment':0,'Cash or Installment':0.5}).fillna(0.5)
df['completion_enc'] = df['Completion Status'].map({'Ready':1,'Off-plan':0}).fillna(0.5)
df['log_price_m']    = np.log(df['price_m'])

print('Ownership distribution:')
print(df['Ownership'].value_counts().to_string())
print()
print('Payment Option distribution:')
print(df['Payment Option'].value_counts().to_string())
print()
print('Completion Status distribution:')
print(df['Completion Status'].value_counts().to_string())

## 5. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(df['price_m'], bins=70, ax=axes[0], color='steelblue', kde=True)
axes[0].axvline(df['price_m'].median(), color='red',    linestyle='--', label=f'Median: {df["price_m"].median():.1f}M')
axes[0].axvline(df['price_m'].mean(),   color='orange', linestyle='--', label=f'Mean: {df["price_m"].mean():.1f}M')
axes[0].set_title('Price Distribution', fontweight='bold')
axes[0].set_xlabel('Price (Million EGP)'); axes[0].legend()

order = df.groupby('prop_type')['price_m'].median().sort_values().index
sns.boxplot(data=df, x='prop_type', y='price_m', ax=axes[1], palette='Set2', order=order, showfliers=False)
axes[1].set_title('Price by Property Type', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
plt.suptitle('Figure 1 — Price Distribution', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('fig1.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
df.groupby('Governorate')['price_m'].median().sort_values().tail(12).plot(kind='barh', ax=axes[0], color='teal')
axes[0].set_title('Median Price by Governorate (Top 12)', fontweight='bold')
axes[0].set_xlabel('Median Price (M EGP)')

df.groupby('City')['price_m'].median().sort_values().tail(12).plot(kind='barh', ax=axes[1], color='#336EA8')
axes[1].set_title('Median Price by City (Top 12)', fontweight='bold')
axes[1].set_xlabel('Median Price (M EGP)')
plt.suptitle('Figure 2 — Location Analysis', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('fig2.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, lbl, order in [
    (axes[0], 'Ownership',        'Ownership',         ['Primary','Resale']),
    (axes[1], 'Completion Status', 'Completion Status', ['Ready','Off-plan']),
    (axes[2], 'Payment Option',    'Payment Option',    ['Cash','Installment','Cash or Installment']),
]:
    valid = [o for o in order if o in df[col].dropna().unique()]
    sns.boxplot(data=df[df[col].isin(valid)], x=col, y='price_m',
                ax=ax, palette='Set2', order=valid, showfliers=False)
    ax.set_title(f'Price by {lbl}', fontweight='bold')
    ax.set_xlabel(''); ax.tick_params(axis='x', rotation=15)
plt.suptitle('Figure 3 — New Features Impact on Price', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('fig3.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Log-transform
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['price_m'],     bins=60, ax=axes[0], color='steelblue', kde=True)
axes[0].set_title(f'Before Log  (Skewness: {df["price_m"].skew():.2f})', fontweight='bold')
axes[0].set_xlabel('Price (Million EGP)')
sns.histplot(df['log_price_m'], bins=60, ax=axes[1], color='teal', kde=True)
axes[1].set_title(f'After Log   (Skewness: {df["log_price_m"].skew():.2f})', fontweight='bold')
axes[1].set_xlabel('log(Price M EGP)')
plt.suptitle('Figure 4 — Log-Transform: Skewness Reduction', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('fig4.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. Label Encoding & Train/Test Split

In [ ]:
le_gc  = LabelEncoder(); df['gc_enc']  = le_gc.fit_transform(df['gov_city'].fillna('Unknown'))
le_cd  = LabelEncoder(); df['cd_enc']  = le_cd.fit_transform(df['city_district'].fillna('Unknown'))

print(f'Gov+City combinations   : {len(le_gc.classes_)}')
print(f'City+District combinations: {len(le_cd.classes_)}')

FEATURES    = ['Built-Up Area (m²)','Bedrooms','gc_enc','cd_enc',
               'ownership_enc','payment_enc','completion_enc']
FEAT_LABELS = ['Area (m²)','Bedrooms','Gov+City','City+District','Ownership','Payment','Completion']

# NOTE: Bathrooms excluded — correlated 0.79 with Bedrooms;
#       1→2 bath jump reflects market tier, not causal per-bathroom effect.
print(f'\nFeatures: {FEATURES}')

# Correlation check
corr = df[FEATURES + ['price_m']].corr()['price_m'].drop('price_m').sort_values(ascending=False)
print(f'\nCorrelation with price_m:')
print(corr.round(3).to_string())

## 7. Stratified Model Training

**Why stratified?**
- Property type is a tier selector, not a numeric feature
- Apartment pricing dynamics differ fundamentally from Villa pricing
- Removes arbitrary label-encoding of property type
- Each model learns type-specific area/location/ownership relationships

**Model selection per type:** train RF and GB → keep the one with higher test R²


In [ ]:
def get_metrics(y_true, y_pred):
    return dict(
        MAE=round(mean_absolute_error(y_true, y_pred), 3),
        RMSE=round(np.sqrt(mean_squared_error(y_true, y_pred)), 3),
        R2=round(r2_score(y_true, y_pred), 4),
        MAPE=round(np.mean(np.abs((y_true - y_pred) / y_true)) * 100, 2),
    )

VALID_TYPES = [t for t, c in df['prop_type'].value_counts().items() if c >= 200]
type_models, type_results = {}, {}
all_true, all_pred = [], []

print(f'{"Type":<14} {"n":>6} {"Best Model":<18} {"MAE":>8} {"R²":>8} {"MAPE":>8} {"CVR²":>8}')
print('─' * 72)

for ptype in VALID_TYPES:
    sub = df[df['prop_type'] == ptype].dropna(subset=FEATURES)
    X = sub[FEATURES]; y_log = sub['log_price_m']; y_raw = sub['price_m']
    X_tr, X_te, y_tr_log, _ = train_test_split(X, y_log, test_size=0.2, random_state=42)
    _,    _,    _,       yte = train_test_split(X, y_raw, test_size=0.2, random_state=42)

    best_model, best_name, best_r2 = None, '', -99
    for name, model in [
        ('Random Forest',
         RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=3, random_state=42, n_jobs=-1)),
        ('Gradient Boosting',
         GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, subsample=0.8, random_state=42)),
    ]:
        model.fit(X_tr, y_tr_log)
        r2 = r2_score(yte.values, np.exp(model.predict(X_te)))
        if r2 > best_r2:
            best_r2, best_name, best_model = r2, name, model

    yp = np.exp(best_model.predict(X_te))
    m  = get_metrics(yte.values, yp)
    cv = cross_val_score(best_model, X_tr, y_tr_log, cv=5, scoring='r2').mean()
    m.update({'CVR2': round(cv, 4), 'count': len(sub), 'model_name': best_name})

    type_models[ptype]  = best_model
    type_results[ptype] = m
    all_true.extend(yte.values)
    all_pred.extend(yp)
    print(f'{ptype:<14} {len(sub):>6,} {best_name:<18} {m["MAE"]:>8.3f} {m["R2"]:>8.4f} {m["MAPE"]:>7.1f}% {cv:>8.4f}')

print('─' * 72)
t, p = np.array(all_true), np.array(all_pred)
g_r2   = round(r2_score(t, p), 4)
g_mape = round(np.mean(np.abs((t - p) / t)) * 100, 2)
g_mae  = round(mean_absolute_error(t, p), 3)
print(f'Global  R²={g_r2}  MAE={g_mae}M  MAPE={g_mape}%')

## 8. Model Comparison

In [ ]:
summary = pd.DataFrame({
    t: {k: v for k, v in r.items() if k not in ('model_name',)}
    for t, r in type_results.items()
}).T

def highlight(s):
    lower_better = s.name in ('MAE', 'RMSE', 'MAPE')
    best = s.min() if lower_better else s.max()
    return ['background-color:#d4edda;font-weight:bold' if v == best else '' for v in s]

print(summary[['count','MAE','RMSE','R2','MAPE','CVR2']].to_string())
summary[['MAE','RMSE','R2','MAPE','CVR2']].astype(float).style.apply(highlight)

## 9. Visualisations

In [ ]:
# Per-type performance
clrs = ['#047954','#336EA8','#C44E52','#D7B119','#6B7280','#8B5CF6','#0FAB7D','#F97316'][:len(VALID_TYPES)]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].barh(VALID_TYPES, [type_results[t]['MAPE'] for t in VALID_TYPES], color=clrs, edgecolor='white')
axes[0].axvline(g_mape, color='red', linestyle='--', label=f'Global {g_mape:.1f}%')
axes[0].set_xlabel('MAPE %'); axes[0].set_title('MAPE (Lower = Better)', fontweight='bold'); axes[0].legend()
for i, t in enumerate(VALID_TYPES):
    axes[0].text(type_results[t]['MAPE'] + 0.3, i, f"{type_results[t]['MAPE']:.1f}%", va='center', fontsize=9)

axes[1].barh(VALID_TYPES, [type_results[t]['R2'] for t in VALID_TYPES], color=clrs, edgecolor='white')
axes[1].axvline(g_r2, color='red', linestyle='--', label=f'Global {g_r2:.3f}')
axes[1].set_xlabel('R²'); axes[1].set_title('R² Score (Higher = Better)', fontweight='bold'); axes[1].legend()
for i, t in enumerate(VALID_TYPES):
    axes[1].text(type_results[t]['R2'] + 0.005, i, f"{type_results[t]['R2']:.3f}", va='center', fontsize=9)

axes[2].barh(VALID_TYPES, [type_results[t]['count'] / 1000 for t in VALID_TYPES], color=clrs, edgecolor='white')
axes[2].set_xlabel('Count (thousands)'); axes[2].set_title('Training Records', fontweight='bold')

plt.suptitle('Figure 5 — Stratified Model Performance per Type', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('fig5.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Actual vs Predicted — all types
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, ptype, color in zip(axes.flat, VALID_TYPES, clrs):
    sub = df[df['prop_type'] == ptype].dropna(subset=FEATURES)
    _, X_te, _, yte = train_test_split(sub[FEATURES], sub['price_m'], test_size=0.2, random_state=42)
    yp  = np.exp(type_models[ptype].predict(X_te))
    lim = [min(yte.min(), yp.min()) - 0.5, max(yte.max(), yp.max()) + 0.5]
    ax.scatter(yte, yp, alpha=0.2, s=7, color=color)
    ax.plot(lim, lim, 'k--', linewidth=1.2)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('Actual (M EGP)', fontsize=9)
    ax.set_ylabel('Predicted (M EGP)', fontsize=9)
    ax.set_title(f'{ptype}\nR²={type_results[ptype]["R2"]:.3f}  MAPE={type_results[ptype]["MAPE"]:.1f}%',
                 fontweight='bold', fontsize=10)
for ax in axes.flat[len(VALID_TYPES):]: ax.set_visible(False)
plt.suptitle('Figure 6 — Actual vs. Predicted (All Types)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('fig6.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Feature importances
tree_types = [t for t in VALID_TYPES if hasattr(type_models[t], 'feature_importances_')][:4]
fig, axes = plt.subplots(1, len(tree_types), figsize=(5 * len(tree_types), 5))
for ax, ptype in zip(axes, tree_types):
    imp = type_models[ptype].feature_importances_
    idx = np.argsort(imp)
    ax.barh([FEAT_LABELS[i] for i in idx], imp[idx],
            color=clrs[VALID_TYPES.index(ptype)], edgecolor='white')
    ax.set_title(ptype, fontweight='bold')
    ax.set_xlabel('Importance Score')
    for i, v in enumerate(imp[idx]):
        ax.text(v + 0.003, i, f'{v:.3f}', va='center', fontsize=9)
plt.suptitle('Figure 7 — Feature Importances', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('fig7.png', dpi=150, bbox_inches='tight'); plt.show()

## 10. Save Model Bundle

In [ ]:
# Build cascading dropdown maps for the Streamlit app
gov_city_map, city_district_map = {}, {}
for gc in le_gc.classes_:
    if not isinstance(gc, str): continue
    parts = gc.split(' — ', 1)
    if len(parts) == 2: gov_city_map.setdefault(parts[0], []).append(gc)
for cd in le_cd.classes_:
    if not isinstance(cd, str): continue
    parts = cd.split(' / ', 1)
    if len(parts) == 2: city_district_map.setdefault(parts[0], []).append(cd)

bundle = {
    'approach'         : 'stratified_dubizzle',
    'type_models'      : type_models,
    'type_results'     : {t: {k: v for k, v in r.items()} for t, r in type_results.items()},
    'le_gc'            : le_gc,
    'le_cd'            : le_cd,
    'features'         : FEATURES,
    'feat_labels'      : FEAT_LABELS,
    'valid_types'      : VALID_TYPES,
    'gov_city_map'     : gov_city_map,
    'city_district_map': city_district_map,
    'gov_cities'       : [g for g in le_gc.classes_ if isinstance(g, str)],
    'city_districts'   : [c for c in le_cd.classes_ if isinstance(c, str)],
    'global_metrics'   : dict(R2=g_r2, MAE=g_mae, MAPE=g_mape),
}
joblib.dump(bundle, 'model_final.pkl')
print(f'✅ model_final.pkl saved')
print(f'   {len(type_models)} stratified models')
print(f'   Global: R²={g_r2}  MAE={g_mae}M  MAPE={g_mape}%')

## 11. Sample Predictions

In [ ]:
def predict_price(area, bedrooms, gov_city, city_district, ownership, payment, completion, prop_type):
    """Predict property price from inputs."""
    b = joblib.load('model_final.pkl')
    if prop_type not in b['type_models']:
        raise ValueError(f'No model for type: {prop_type}')

    gc_enc   = b['le_gc'].transform([gov_city])[0]
    cd_enc   = b['le_cd'].transform([city_district])[0]
    own_enc  = {'Primary': 1, 'Resale': 0}.get(ownership, 0.5)
    pay_enc  = {'Cash': 1, 'Installment': 0, 'Cash or Installment': 0.5}.get(payment, 0.5)
    comp_enc = {'Ready': 1, 'Off-plan': 0}.get(completion, 0.5)

    X = pd.DataFrame([{
        'Built-Up Area (m²)': area, 'Bedrooms': bedrooms,
        'gc_enc': gc_enc, 'cd_enc': cd_enc,
        'ownership_enc': own_enc, 'payment_enc': pay_enc, 'completion_enc': comp_enc,
    }])
    return np.exp(b['type_models'][prop_type].predict(X)[0])

# Sample predictions
examples = [
    (140,  3, 'Cairo — New Cairo',     'New Cairo / 5th Settlement',   'Primary',   'Cash',        'Ready',    'Apartment',   '3BR Apt, 5th Settlement, New Cairo'),
    (300,  5, 'Cairo — New Cairo',     'New Cairo / 5th Settlement',   'Primary',   'Cash',        'Ready',    'Villa',       '5BR Villa, 5th Settlement'),
    (160,  3, 'Giza — Sheikh Zayed',   'Sheikh Zayed / El Khamayel Compound', 'Primary', 'Installment', 'Off-plan', 'Town House', '3BR TH, Sheikh Zayed (Off-plan)'),
    (90,   2, 'Giza — Sheikh Zayed',   'Sheikh Zayed / Green Revolution Compound', 'Primary', 'Installment', 'Off-plan', 'Apartment', '2BR Apt, Sheikh Zayed (Off-plan)'),
    (250,  4, 'Matruh — North Coast',  'North Coast / Ras Al Hekma',   'Primary',   'Installment', 'Off-plan', 'Villa',       '4BR Villa, Ras Al Hekma (Off-plan)'),
    (70,   1, 'Cairo — New Cairo',     'New Cairo / 5th Settlement',   'Resale',    'Cash',        'Ready',    'Studio',      'Studio, 5th Settlement (Resale)'),
]
print(f'{"Property":<45} {"Predicted Price":>15}  {"(EGP)":>14}')
print('─' * 78)
for area, beds, gc, cd, own, pay, comp, ptype, label in examples:
    try:
        p = predict_price(area, beds, gc, cd, own, pay, comp, ptype)
        print(f'{label:<45} {p:>8.2f}M EGP  {p*1e6:>14,.0f}')
    except Exception as e:
        print(f'{label:<45} Error: {e}')

## 12. Conclusions

### Final Results

| Type | Model | n | MAE | R² | MAPE |
|------|-------|---|-----|-----|------|
| Apartment | Random Forest | 43,399 | 2.056M | 0.5336 | 28.1% |
| Villa | Random Forest | 13,975 | 7.933M | 0.6019 | 29.6% |
| Town House | Gradient Boosting | 5,565 | 3.816M | 0.5611 | 24.1% |
| Twin House | Gradient Boosting | 3,387 | 4.512M | 0.6328 | 23.2% |
| Duplex | Gradient Boosting | 3,435 | 3.322M | 0.4098 | 31.4% |
| Penthouse | Gradient Boosting | 2,577 | 3.373M | 0.5029 | 23.6% |
| Studio | Random Forest | 994 | 1.224M | 0.5882 | 25.0% |
| **Global** | — | **74,240** | **3.532M** | **0.7576** | **27.87%** |

### Key Findings
- **City+District** (1,473 combinations) is the most important feature across all types
- **Completion Status** and **Ownership** add significant value not in previous datasets  
- **Log-transform** essential: skewness 2.63 → 0.08, cuts MAPE by ~30 points
- **Stratified approach**: each type has distinct price dynamics
- **Bathrooms removed**: r=0.79 with bedrooms; jump reflects market tier not causal effect

### vs. Kaggle Baseline
| Metric | Kaggle | data.xlsx | Dubizzle (Final) |
|--------|--------|-----------|-----------------|
| R² | 0.49 | 0.69 | **0.758** |
| MAPE | 57% | 33% | **27.87%** |
| MAE | 4.55M | 4.17M | **3.53M** |

### Future Work
- XGBoost + Optuna tuning → target R² > 0.85
- Floor number + finishing quality features
- Geographic coordinates (lat/lon) for spatial regression
- Monthly automated re-scrape to track price trends
